In [1]:
## fetching the dataset 
import pandas as pd 

df = pd.read_csv('data/bank.csv')

## display the dataset 
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [2]:
import warnings
warnings.filterwarnings('ignore')

## Splitting features and Target

In [3]:
X = df.drop(['duration','deposit'],axis=1)
y = df['deposit']

In [4]:
## Segregating numerical and categorical columns
num_columns = X.select_dtypes(include='number').columns
cat_columns = X.select_dtypes(include='object').columns

In [5]:
## Creating pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer([
    ('num_pipeline',num_pipeline,num_columns),
    ('cat_pipeline',cat_pipeline,cat_columns)
])

In [6]:
## Splitting the data using train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.30,random_state=42,stratify=df['deposit'])

In [7]:
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(preprocessor.transform(X_test),columns=preprocessor.get_feature_names_out())

In [8]:
X_train.head()

,num_pipeline__age,num_pipeline__balance,num_pipeline__day,num_pipeline__campaign,num_pipeline__pdays,num_pipeline__previous,cat_pipeline__job_admin.,cat_pipeline__job_blue-collar,cat_pipeline__job_entrepreneur,cat_pipeline__job_housemaid,...,cat_pipeline__month_jun,cat_pipeline__month_mar,cat_pipeline__month_may,cat_pipeline__month_nov,cat_pipeline__month_oct,cat_pipeline__month_sep,cat_pipeline__poutcome_failure,cat_pipeline__poutcome_other,cat_pipeline__poutcome_success,cat_pipeline__poutcome_unknown
0,1.507850,-0.498913,1.571827,-0.176950,-0.481859,-0.355477,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.851198,-0.346542,1.690409,-0.533542,-0.481859,-0.355477,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.598443,0.928809,0.623173,-0.533542,-0.481859,-0.355477,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.682694,-0.397220,-1.511299,-0.533542,2.303242,1.781271,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.261436,-0.498913,1.334664,-0.533542,0.219027,0.926572,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [9]:
## Using label encoder to transform target
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [10]:
## Evaluating the model
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score
import numpy as np

def eval_model(true,predict,proba=None):
    accuracy = accuracy_score(true,predict)
    confi_matrix = confusion_matrix(true,predict)
    precision = precision_score(true,predict)
    recall = recall_score(true,predict)
    f1 = f1_score(true,predict)

    if proba is not None:
        roc_auc = roc_auc_score(true,proba)
    else:
        roc_auc = np.nan

    return accuracy,confi_matrix,precision,recall,f1,roc_auc

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

models = {
    "LogisticRegression":LogisticRegression(),
    "DecisionTreeClassifier":DecisionTreeClassifier(),
    "RandomForestClassifier":RandomForestClassifier(),
    "GradientBoostingClassifier":GradientBoostingClassifier(),
    "AdaBoostClassifier":AdaBoostClassifier(),
    "SVC":SVC(),
    "XGBClassifier":XGBClassifier(),
    "CatBoostClassifier":CatBoostClassifier()
}


param_distributions = {

    "LogisticRegression": {
        "C": loguniform(1e-3, 1e2),
        "penalty": ["l1", "l2"],
        "solver": ["liblinear", "saga"],
        "class_weight": [None, "balanced"],
        "max_iter": [1000, 2000, 3000]
    },

    "DecisionTreeClassifier": {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 5, 10, 15, 20, 30],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 20),
        "max_features": [None, "sqrt", "log2"],
        "class_weight": [None, "balanced"]
    },

    "RandomForestClassifier": {
        "n_estimators": randint(100, 800),
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 10, 20, 30, 40],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 15),
        "max_features": ["sqrt", "log2", None],
        "bootstrap": [True, False],
        "class_weight": [None, "balanced", "balanced_subsample"]
    },

    "GradientBoostingClassifier": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(0.01, 0.3),
        "max_depth": randint(1, 6),
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 20),
        "subsample": uniform(0.6, 0.4),
        "max_features": [None, "sqrt", "log2"]
    },

    "AdaBoostClassifier": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(0.01, 2.0)
    },

    "SVC": {
        "C": loguniform(1e-3, 1e2),
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"],
        "class_weight": [None, "balanced"],
        "degree": randint(2, 5)
    },

    "XGBClassifier": {
        "n_estimators": randint(100, 800),
        "max_depth": randint(3, 11),
        "learning_rate": loguniform(0.01, 0.3),
        "min_child_weight": randint(1, 15),
        "gamma": uniform(0, 5),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "reg_alpha": loguniform(1e-4, 10),
        "reg_lambda": loguniform(1e-3, 20),
        "scale_pos_weight": [1, 2, 3, 4, 5]
    },

    "CatBoostClassifier": {
        "iterations": randint(200, 1000),
        "depth": randint(4, 11),
        "learning_rate": loguniform(0.01, 0.3),
        "l2_leaf_reg": loguniform(1, 10),
        "random_strength": uniform(0, 2),
        "bagging_temperature": uniform(0, 2),
        "border_count": [32, 64, 128, 254],
        "auto_class_weights": [None, "Balanced", "SqrtBalanced"]
    }
}

report={}

best_model = None
best_score = -1
best_params = None

for model_name, model in models.items():
    print(f"{model_name} evaluation started..")
    search = RandomizedSearchCV(estimator=model, param_distributions=param_distributions[model_name],
                            cv=5,n_iter=50,verbose=1,n_jobs=-1,scoring='roc_auc', random_state=42)

    search.fit(X_train,y_train)

    cv_score = search.best_score_
    tuned_model = search.best_estimator_
    tuned_params = search.best_params_

    print(f"{model_name} best score: {cv_score} and its best params: {tuned_params}")

    if cv_score > best_score:
        best_score = cv_score
        best_model = tuned_model
        best_params = tuned_params

    y_pred = search.predict(X_test)

    if hasattr(model,'predict_proba'):
        y_proba = search.predict_proba(X_test)[:, 1]
    elif hasattr(model,'decision_function'):
        y_proba = search.decision_function(X_test)

    accuracy,confi_matrix,precision,recall,f1,roc_auc = eval_model(y_test,y_pred,proba=y_proba)

    report[model_name] = {
        "accuracy": accuracy,
        "recall": recall,
        "precision": precision,
        "f1": f1,
        'roc_auc': roc_auc
    }
    print()
    print(f"{model_name}_confusion matrix: \n{str(confi_matrix)}")
    print('='*15)

report_df = pd.DataFrame(report).T.sort_values(by='roc_auc',ascending=False).reset_index().rename(columns={'index': 'Model'})
print(f"Best Model found: {best_model} with best score: {best_score} and params: {best_params}")
print()
report_df.head(10)

LogisticRegression evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
LogisticRegression best score: 0.7620701323068231 and its best params: {'C': np.float64(0.14445251022763064), 'class_weight': None, 'max_iter': 3000, 'penalty': 'l1', 'solver': 'liblinear'}

LogisticRegression_confusion matrix: 
[[1442  320]
 [ 670  917]]
DecisionTreeClassifier evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
DecisionTreeClassifier best score: 0.7433177320657428 and its best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 18, 'min_samples_split': 27}

DecisionTreeClassifier_confusion matrix: 
[[1317  445]
 [ 571 1016]]
RandomForestClassifier evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
RandomForestClassifier best score: 0.7885882741339899 and its best params: {'bootstrap': True, 'class_weight': 'balanced_subsample', 'criterion':

,Model,accuracy,recall,precision,f1,roc_auc
0,XGBClassifier,0.621678,0.900441,0.563042,0.692848,0.795430
1,CatBoostClassifier,0.737832,0.632010,0.773323,0.695562,0.792660
2,GradientBoostingClassifier,0.741117,0.630120,0.781250,0.697593,0.791518
3,RandomForestClassifier,0.724993,0.634531,0.747033,0.686201,0.786978
4,SVC,0.730069,0.586641,0.789652,0.673174,0.780790
5,AdaBoostClassifier,0.710361,0.603025,0.737857,0.663662,0.773389
6,LogisticRegression,0.704389,0.577820,0.741310,0.649433,0.761816
7,DecisionTreeClassifier,0.696626,0.640202,0.695414,0.666667,0.747149


In [12]:
## Taining the dataset on FE dataset 
df2 = pd.read_csv('data/fe_bank.csv')
df2.head()

,age,job,marital,education,default,housing,loan,contact,day,month,pdays,poutcome,deposit,balance_log,campaign_log,previous_log
0,59,admin.,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.759614,0.693147,0.0
1,56,admin.,married,secondary,no,no,no,unknown,5,may,-1,unknown,yes,3.828641,0.693147,0.0
2,41,technician,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.147559,0.693147,0.0
3,55,services,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.814803,0.693147,0.0
4,54,admin.,married,tertiary,no,no,no,unknown,5,may,-1,unknown,yes,5.220356,1.098612,0.0


In [14]:
X2 = df2.drop('deposit',axis=1)
y2 = df2['deposit']

In [15]:
x2_num_columns = X2.select_dtypes(include='number').columns
x2_cat_columns = X2.select_dtypes(include='object').columns

In [16]:
num_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer([
    ('num_pipeline',num_pipeline,x2_num_columns),
    ('cat_pipeline',cat_pipeline,x2_cat_columns)
])

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X2,y2,test_size=0.30,random_state=42,stratify=df2['deposit'])

In [18]:
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(preprocessor.transform(X_test),columns=preprocessor.get_feature_names_out())

In [19]:
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [20]:
models = {
    "LogisticRegression":LogisticRegression(),
    "DecisionTreeClassifier":DecisionTreeClassifier(),
    "RandomForestClassifier":RandomForestClassifier(),
    "GradientBoostingClassifier":GradientBoostingClassifier(),
    "AdaBoostClassifier":AdaBoostClassifier(),
    "SVC":SVC(),
    "XGBClassifier":XGBClassifier(),
    "CatBoostClassifier":CatBoostClassifier()
}


param_distributions = {

    "LogisticRegression": {
        "C": loguniform(1e-3, 1e2),
        "penalty": ["l1", "l2"],
        "solver": ["liblinear", "saga"],
        "class_weight": [None, "balanced"],
        "max_iter": [1000, 2000, 3000]
    },

    "DecisionTreeClassifier": {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 5, 10, 15, 20, 30],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 20),
        "max_features": [None, "sqrt", "log2"],
        "class_weight": [None, "balanced"]
    },

    "RandomForestClassifier": {
        "n_estimators": randint(100, 800),
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 10, 20, 30, 40],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 15),
        "max_features": ["sqrt", "log2", None],
        "bootstrap": [True, False],
        "class_weight": [None, "balanced", "balanced_subsample"]
    },

    "GradientBoostingClassifier": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(0.01, 0.3),
        "max_depth": randint(1, 6),
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 20),
        "subsample": uniform(0.6, 0.4),
        "max_features": [None, "sqrt", "log2"]
    },

    "AdaBoostClassifier": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(0.01, 2.0)
    },

    "SVC": {
        "C": loguniform(1e-3, 1e2),
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"],
        "class_weight": [None, "balanced"],
        "degree": randint(2, 5)
    },

    "XGBClassifier": {
        "n_estimators": randint(100, 800),
        "max_depth": randint(3, 11),
        "learning_rate": loguniform(0.01, 0.3),
        "min_child_weight": randint(1, 15),
        "gamma": uniform(0, 5),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "reg_alpha": loguniform(1e-4, 10),
        "reg_lambda": loguniform(1e-3, 20),
        "scale_pos_weight": [1, 2, 3, 4, 5]
    },

    "CatBoostClassifier": {
        "iterations": randint(200, 1000),
        "depth": randint(4, 11),
        "learning_rate": loguniform(0.01, 0.3),
        "l2_leaf_reg": loguniform(1, 10),
        "random_strength": uniform(0, 2),
        "bagging_temperature": uniform(0, 2),
        "border_count": [32, 64, 128, 254],
        "auto_class_weights": [None, "Balanced", "SqrtBalanced"]
    }
}

report={}

best_model = None
best_score = -1
best_params = None

for model_name, model in models.items():
    print(f"{model_name} evaluation started..")
    search = RandomizedSearchCV(estimator=model, param_distributions=param_distributions[model_name],
                            cv=5,n_iter=50,verbose=1,n_jobs=-1,scoring='roc_auc', random_state=42)

    search.fit(X_train,y_train)

    cv_score = search.best_score_
    tuned_model = search.best_estimator_
    tuned_params = search.best_params_

    print(f"{model_name} best score: {cv_score} and its best params: {tuned_params}")

    if cv_score > best_score:
        best_score = cv_score
        best_model = tuned_model
        best_params = tuned_params

    y_pred = search.predict(X_test)

    if hasattr(model,'predict_proba'):
        y_proba = search.predict_proba(X_test)[:, 1]
    elif hasattr(model,'decision_function'):
        y_proba = search.decision_function(X_test)

    accuracy,confi_matrix,precision,recall,f1,roc_auc = eval_model(y_test,y_pred,proba=y_proba)

    report[model_name] = {
        "accuracy": accuracy,
        "recall": recall,
        "precision": precision,
        "f1": f1,
        'roc_auc': roc_auc
    }
    print()
    print(f"{model_name}_confusion matrix: \n{str(confi_matrix)}")
    print('='*15)

report_df = pd.DataFrame(report).T.sort_values(by='roc_auc',ascending=False).reset_index().rename(columns={'index': 'Model'})
print(f"Best Model found: {best_model} with best score: {best_score} and params: {best_params}")
print()
report_df.head(10)

LogisticRegression evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
LogisticRegression best score: 0.7645495211989639 and its best params: {'C': np.float64(0.14445251022763064), 'class_weight': None, 'max_iter': 3000, 'penalty': 'l1', 'solver': 'liblinear'}

LogisticRegression_confusion matrix: 
[[1424  338]
 [ 647  940]]
DecisionTreeClassifier evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
DecisionTreeClassifier best score: 0.738725103617204 and its best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 24}

DecisionTreeClassifier_confusion matrix: 
[[1343  419]
 [ 706  881]]
RandomForestClassifier evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
RandomForestClassifier best score: 0.7884502475432 and its best params: {'bootstrap': True, 'class_weight': None, 'criterion': 'log_loss', 'max_depth': 2

,Model,accuracy,recall,precision,f1,roc_auc
0,XGBClassifier,0.623768,0.901071,0.564548,0.694175,0.795145
1,GradientBoostingClassifier,0.739027,0.630750,0.776571,0.696106,0.793385
2,CatBoostClassifier,0.738429,0.630750,0.775368,0.695622,0.791986
3,RandomForestClassifier,0.729770,0.613737,0.769352,0.682790,0.787837
4,SVC,0.732457,0.608696,0.778405,0.683168,0.782014
5,AdaBoostClassifier,0.710361,0.603025,0.737857,0.663662,0.773389
6,LogisticRegression,0.705882,0.592313,0.735524,0.656195,0.761480
7,DecisionTreeClassifier,0.664079,0.555135,0.677692,0.610322,0.713349
